# GNN Spam Detection — 전략 탐색 (Legacy)

Strategy A (상위 상품) / Strategy C (시간 구간) 탐색 실험 아카이브.  
최종 파이프라인(ACO + 커스텀 엣지)으로 수렴하기 전 비교 실험.

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, average_precision_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
df = pd.read_csv('yelpzip.csv', index_col=0)
df['label'] = df['label'].map({-1: 1, 1: 0})
df['date'] = pd.to_datetime(df['date'])

user_review_cnt = df.groupby('user_id').size()
prod_review_cnt = df.groupby('prod_id').size()
print(f'Total: {len(df):,} | Spam: {df["label"].mean():.2%}')

In [ ]:
from itertools import combinations

def build_edges(df, group_col, max_per_group=30):
    src, dst = [], []
    for _, grp in df.groupby(group_col)['node_id']:
        nodes = grp.tolist()
        if len(nodes) < 2:
            continue
        if len(nodes) > max_per_group:
            nodes = np.random.choice(nodes, max_per_group, replace=False).tolist()
        for u, v in combinations(nodes, 2):
            src += [u, v]; dst += [v, u]
    return src, dst


class GCN(torch.nn.Module):
    def __init__(self, in_ch, hidden=64, dropout=0.5):
        super().__init__()
        self.conv1 = GCNConv(in_ch, hidden)
        self.conv2 = GCNConv(hidden, hidden)
        self.lin   = torch.nn.Linear(hidden, 2)
        self.drop  = dropout

    def forward(self, data):
        x, ei = data.x, data.edge_index
        x = F.dropout(F.relu(self.conv1(x, ei)), p=self.drop, training=self.training)
        x = F.dropout(F.relu(self.conv2(x, ei)), p=self.drop, training=self.training)
        return self.lin(x)


def run_pipeline(raw_sub, name, epochs=100):
    print(f"\n{'='*50}\n  {name}\n{'='*50}")
    sub = raw_sub.reset_index(drop=True).copy()
    sub['node_id'] = range(len(sub))
    sub['month']   = sub['date'].dt.month
    sub['user_cnt'] = sub['user_id'].map(user_review_cnt)
    sub['prod_cnt'] = sub['prod_id'].map(prod_review_cnt)
    sub['text_len'] = sub['text'].str.len()
    sub['prod_rating'] = sub['prod_id'].astype(str) + '_' + sub['rating'].astype(str)

    np.random.seed(42)
    rur = build_edges(sub, 'user_id')
    rsr = build_edges(sub, 'prod_rating')
    ei  = torch.tensor([rur[0]+rsr[0], rur[1]+rsr[1]], dtype=torch.long)
    print(f'R-U-R: {len(rur[0])//2:,} | R-S-R: {len(rsr[0])//2:,} | Total: {ei.shape[1]//2:,}')

    feature_cols = ['rating', 'user_cnt', 'prod_cnt', 'text_len', 'month']
    X = StandardScaler().fit_transform(sub[feature_cols].values)
    y = sub['label'].values
    idx = np.arange(len(sub))
    tr_idx, te_idx = train_test_split(idx, test_size=0.2, stratify=y, random_state=42)
    tr_mask = torch.zeros(len(sub), dtype=torch.bool); tr_mask[tr_idx] = True
    te_mask = torch.zeros(len(sub), dtype=torch.bool); te_mask[te_idx] = True

    data = Data(x=torch.tensor(X, dtype=torch.float),
                edge_index=ei,
                y=torch.tensor(y, dtype=torch.long),
                train_mask=tr_mask, test_mask=te_mask).to(DEVICE)

    n0, n1 = (y==0).sum(), (y==1).sum()
    cw = torch.tensor([1.0, n0/n1], dtype=torch.float).to(DEVICE)
    model = GCN(in_ch=len(feature_cols)).to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
    history = {'loss': [], 'pr_auc': [], 'macro_f1': []}

    for ep in range(1, epochs+1):
        model.train(); opt.zero_grad()
        out  = model(data)
        loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask], weight=cw)
        loss.backward(); opt.step()

        model.eval()
        with torch.no_grad():
            prob = F.softmax(model(data), dim=1)[:,1].cpu().numpy()
            pred = model(data).argmax(dim=1).cpu().numpy()
        y_te   = y[te_mask.cpu()]
        pr_auc = average_precision_score(y_te, prob[te_mask.cpu()])
        mf1    = f1_score(y_te, pred[te_mask.cpu()], average='macro', zero_division=0)
        history['loss'].append(loss.item())
        history['pr_auc'].append(pr_auc)
        history['macro_f1'].append(mf1)
        if ep % 10 == 0:
            print(f'Epoch {ep:3d} | Loss {loss.item():.4f} | PR-AUC {pr_auc:.4f} | F1 {mf1:.4f}')

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, key, color in zip(axes, ['loss', 'pr_auc', 'macro_f1'],
                               ['#4C72B0', '#55A868', '#DD8452']):
        ax.plot(history[key], color=color)
        ax.set_title(key, fontweight='bold'); ax.set_xlabel('Epoch')
    plt.suptitle(name, fontweight='bold'); plt.tight_layout(); plt.show()
    print(f'[{name}] PR-AUC: {history["pr_auc"][-1]:.4f} | F1: {history["macro_f1"][-1]:.4f}')
    return history

## Strategy A — 상위 상품 (Top 13 Products)

In [ ]:
top_prods = prod_review_cnt.nlargest(13).index
strat_A = df[df['prod_id'].isin(top_prods)].copy()
print(f'A — nodes: {len(strat_A):,} | spam: {strat_A["label"].mean():.2%}')
hist_A = run_pipeline(strat_A, 'Strategy A')

## Strategy C — 시간 구간 (2012-01 ~ 2012-06)

In [ ]:
strat_C = df[(df['date'] >= '2012-01-01') & (df['date'] <= '2012-06-30')].copy()
print(f'C — nodes: {len(strat_C):,} | spam: {strat_C["label"].mean():.2%}')
hist_C = run_pipeline(strat_C, 'Strategy C')

## 결과 비교

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, key, title in zip(axes, ['pr_auc', 'macro_f1'], ['PR-AUC', 'Macro F1']):
    ax.plot(hist_A[key], label='Strategy A', color='#4C72B0', alpha=0.8)
    ax.plot(hist_C[key], label='Strategy C', color='#55A868', alpha=0.8)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.legend(frameon=False)
plt.suptitle('Sampling Strategy Comparison (A vs C)', fontweight='bold')
plt.tight_layout()
plt.show()

print('\n=== Final Performance ===')
for name, hist in [('Strategy A', hist_A), ('Strategy C', hist_C)]:
    print(f'{name}: PR-AUC={hist["pr_auc"][-1]:.4f} | F1={hist["macro_f1"][-1]:.4f}')